## a) Prove

In [9]:
import sqlite3
import pandas as pd

In [ ]:
# 1. Connect DB and Load
conn = sqlite3.connect("wiki_diseases.db")
articles = pd.read_sql_query("SELECT * FROM articles", conn)
conn.close()

# 2. Output
print(f"Total rows: {len(articles)}")
print(f"Columns: {articles.columns.tolist()}")

Total rows: 233
Columns: ['title', 'main_text', 'name', 'url', 'datePublished', 'dateModified', 'headline']


## b) Definition of Tokenization and Preprocessing Pipeline

In [4]:
import re
from nltk.corpus import stopwords

In [13]:
# Modified tokenize func
def tokenize(text):
    return re.findall(r'[A-Za-z]+', text)

# Modified stop words func
stop_words = set(stopwords.words('english'))
def remove_stop(tokens):
    return [t for t in tokens if t.lower() not in stop_words]

include_stop_words = {'dear', 'regards', 'must', 'would', 'also'}
exclude_stop_words = {'against'}

stop_words |= include_stop_words
stop_words -= exclude_stop_words

# Pipeline list
pipeline = [str.lower, tokenize, remove_stop]

# Prepare func
def prepare(text, pipeline):
    tokens = text
    for transform in pipeline:
        tokens = transform(tokens)
    return tokens

## c) Apply the function “prepare()”

In [16]:
# add 'tokens'
articles['tokens'] = articles['main_text'].apply(prepare, pipeline=pipeline)

## d) Apply the function “len()”

In [17]:
# add 'num_tokens'
articles['num_tokens'] = articles['tokens'].apply(len)

## e) Show that the DataFrame

In [23]:
# e) Output (random 2 rows)
print(articles[['tokens', 'num_tokens']].sample(2))

                                                tokens  num_tokens
45   [wikipedia, free, encyclopediachronic, disease...        8380
126  [wikipedia, free, encyclopediasymptoms, simila...         486


## f) Word Frequencies

In [39]:
from collections import Counter ###

def count_words(df, column='tokens', preprocess=None, min_freq=2):

    # process tokens and update counter
    def update(doc):
        tokens = doc if preprocess is None else preprocess(doc)
        counter.update(tokens)

    # create counter and run through all data
    counter = Counter()
    df[column].map(update)

    # transform counter into data frame
    freq_df = pd.DataFrame.from_dict(counter, orient='index', columns=['freq'])
    freq_df = freq_df.query('freq >= @min_freq')
    freq_df.index.name = 'token'
    
    return freq_df.sort_values('freq', ascending=False)

In [40]:
freq_df = count_words(articles)
freq_df.head(5)

,freq
token,
doi,16982
pmid,16873
disease,8776
may,8611
pmc,7316


## g) IDF

In [43]:
def compute_idf(df, column='tokens', preprocess=None, min_df=2):

    def update(doc):
        tokens = doc if preprocess is None else preprocess(doc)
        counter.update(set(tokens))

    # count tokens
    counter = Counter()
    df[column].map(update)    

    # create data frame and compute idf
    idf_df = pd.DataFrame.from_dict(counter, orient='index', columns=['df'])
    idf_df = idf_df.query('df >= @min_df') #document frequency (df), not dataframe df
    idf_df['idf'] = np.log(len(df)/idf_df['df'])+0.1
    idf_df.index.name = 'token'
    return idf_df

In [44]:
idf_df = compute_idf(articles)
idf_df.head(5)

,df,idf
token,,
alikhan,2,4.857891
teeth,21,2.506516
marketclinical,8,3.471597
dermatosishypopigmentedtinea,16,2.778450
care,193,0.288348


## h) Perform a left join on “freq_df” with “idf_df” 

In [45]:
freq_df = freq_df.join(idf_df)

freq_df.head()

,freq,df,idf
token,,,
doi,16982,232.0,0.104301
pmid,16873,232.0,0.104301
disease,8776,229.0,0.117316
may,8611,233.0,0.100000
pmc,7316,225.0,0.134938


## i) Calculate the term frequency-inverse document frequency (TF-IDF) values

In [46]:
freq_df['tfidf'] = freq_df['freq'] * freq_df['idf']

freq_df.head()

,freq,df,idf,tfidf
token,,,,
doi,16982,232.0,0.104301,1771.240973
pmid,16873,232.0,0.104301,1759.872155
disease,8776,229.0,0.117316,1029.569165
may,8611,233.0,0.100000,861.100000
pmc,7316,225.0,0.134938,987.206784


## j) Sort the DataFrame “freq_df” based on the values in the “tfidf” column

In [47]:
top_10_tfidf = freq_df.sort_values(by='tfidf', ascending=False).head(10)
print(top_10_tfidf)

           freq     df       idf        tfidf
token                                        
cancer     4756  104.0  0.906648  4312.015769
virus      1981   90.0  1.051229  2082.484220
diabetes   2086  103.0  0.916309  1911.421545
disorder   2559  125.0  0.722725  1849.452549
doi       16982  232.0  0.104301  1771.240973
adhd        692   20.0  2.555306  1768.271877
pmid      16873  232.0  0.104301  1759.872155
autism      735   26.0  2.292942  1685.312308
sleep      1465   84.0  1.120222  1641.124724
ebola       412    5.0  3.941601  1623.939423


## k) Are these returned rows more meaningful?

Answer: The TF-IDF results are more meaningful than the 2b list because they emphasize specific keywords rather than generic terms like "disease" that appear in every document. By filtering out these common words, TF-IDF effectively identifies the unique vocabulary that defines the core subject of each article.